In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/elkanahrutto/redmi6-feedback-analysis/redmi6.csv


In [2]:
#importing necessary packages

import tensorflow as tf
import re
import pandas as pd
import numpy as np

In [3]:
#loading data

path = "/kaggle/input/datasets/elkanahrutto/redmi6-feedback-analysis/redmi6.csv"

df = pd.read_csv(path, encoding="cp1252")

df.head()

,Review Title,Customer name,Rating,Date,Category,Comments,Useful
0,Another Midrange killer Smartphone by Xiaomi,Rishikumar Thakur,4.0 out of 5 stars,on 1 October 2018,Display,Another Midrange killer Smartphone by Xiaomi\n...,
1,vry small size mobile,Raza ji,3.0 out of 5 stars,on 15 September 2018,Others,All ok but vry small size mobile,7 people found this helpful
2,Full display not working in all application.,Vaibhav Patel,3.0 out of 5 stars,on 18 September 2018,Others,Quite good,7 people found this helpful
3,Value for Money,Amazon Customer,5.0 out of 5 stars,on 28 September 2018,Display,Redmi has always have been the the king of bud...,2 people found this helpful
4,Not worth for the money,Sudhakaran Wadakkancheri,2.0 out of 5 stars,on 18 September 2018,Others,worst product from MI. I am a hardcore fan of ...,6 people found this helpful


In [4]:
#Preprocessing(joining relevant columns)

df.columns = df.columns.str.lower()
df_columns = df.columns.to_list()

search_words = ['review', 'comment', 'feedback']
review_columns = []

for word in search_words:
    for col in df_columns:
        if word in col:
            review_columns.append(col)

def merge_columns(row, columns):
    return ' '.join(str(row[col]) for col in columns)

df["Merge_Column"] = df.apply(merge_columns, axis=1, args=(review_columns,))
df.size

2240

In [5]:

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

def analyze_sentiment(text):
    if isinstance(text, tuple):
        text = ' '.join(str(t) for t in text)

    sentiment = SentimentIntensityAnalyzer().polarity_scores(text)

    if sentiment['compound'] > 0.05:
        return f"Positive"
    elif sentiment[ 'compound'] < -0.05:
        return f"Negative"
    else:
        return f"Neutral"

In [6]:
df['sentiments_nltk'] = df['Merge_Column'].apply(analyze_sentiment)
df.size

2520

## RoBERTa Model

In [7]:
#importing neceassary libraries for model transformation
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import pipeline
from scipy.special import softmax

In [8]:
#pull out model weights from pretrained model

MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
#testing the model with a reveiw example

example = "I hate is oatmeal, its very mushy and not tasty at all."
encoded_text = tokenizer(example, return_tensors= 'pt')
output = model(**encoded_text)
scores = output[0][0].detach().numpy()
scores = softmax(scores)
scores_dict = {
    'neg' : scores[0],
    'neu' : scores[1],
    'pos' : scores[2]
}

scores_dict


{'neg': np.float32(0.982057),
 'neu': np.float32(0.01521297),
 'pos': np.float32(0.0027301076)}

In [10]:
import torch
import operator
def polarity_scores(text):
    tokenized_sentence = tokenizer(text, padding=True, truncation=True, max_length=512, \
        return_tensors='pt')
    input_ids = tokenized_sentence['input_ids']
    position_ids = torch.arange(len(input_ids[0])).unsqueeze(0)
    output = model(**tokenized_sentence)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
    'negative' : scores[0],
    'neutral'  : scores[1],
    'positive' : scores[2],
    }

    compound = (max(scores_dict.items(), key=operator.itemgetter(1))[0]).upper()

    return compound


In [11]:
#new_df2 = new_df2.drop('sentiments', axis=1)
#new_df2.size

In [12]:
# from nltk.corpus import stopwords

# df_bert = pd.DataFrame(new_df2.astype(str).apply(' '.join, axis=1), columns=['merged_review'])
# df_bert['merged_review'] = df_bert['merged_review'].str.lower()
# stop_words = set(stopwords.words('english'))
# df_bert['merged_reveiw'] =df_bert['merged_review'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

# df_bert.size

In [13]:
# for index, row in df.itertuples():
#     sentiment = polarity_scores(getattr(row, reveiw_columns))
#     new_df2.loc[index,'sentiments'] = sentiment.upper()
# new_df2

df['sentiments_bert'] = df['Merge_Column'].apply(polarity_scores)
df

,review title,customer name,rating,date,category,comments,useful,Merge_Column,sentiments_nltk,sentiments_bert
0,Another Midrange killer Smartphone by Xiaomi,Rishikumar Thakur,4.0 out of 5 stars,on 1 October 2018,Display,Another Midrange killer Smartphone by Xiaomi\n...,,Another Midrange killer Smartphone by Xiaomi A...,Positive,POSITIVE
1,vry small size mobile,Raza ji,3.0 out of 5 stars,on 15 September 2018,Others,All ok but vry small size mobile,7 people found this helpful,vry small size mobile All ok but vry small siz...,Positive,NEUTRAL
2,Full display not working in all application.,Vaibhav Patel,3.0 out of 5 stars,on 18 September 2018,Others,Quite good,7 people found this helpful,Full display not working in all application. Q...,Positive,POSITIVE
3,Value for Money,Amazon Customer,5.0 out of 5 stars,on 28 September 2018,Display,Redmi has always have been the the king of bud...,2 people found this helpful,Value for Money Redmi has always have been the...,Positive,POSITIVE
4,Not worth for the money,Sudhakaran Wadakkancheri,2.0 out of 5 stars,on 18 September 2018,Others,worst product from MI. I am a hardcore fan of ...,6 people found this helpful,Not worth for the money worst product from MI....,Negative,NEGATIVE
...,...,...,...,...,...,...,...,...,...,...
275,"Cemera quality,face unlock most important in t...",Rahul,5.0 out of 5 stars,on 19 September 2018,Others,"I like This Phone, Awesome look and design.\nI...",NaN,"Cemera quality,face unlock most important in t...",Positive,POSITIVE
276,Mi is best phone,Sunil Soni,4.0 out of 5 stars,on 18 September 2018,Others,Product is avasome but invoice is note include...,NaN,Mi is best phone Product is avasome but invoic...,Positive,POSITIVE
277,Its a OK Phone,D.C.Padhi,3.0 out of 5 stars,on 15 September 2018,Battery,"Redmi Note4, Note5, now 6pro..It seems the old...",NaN,"Its a OK Phone Redmi Note4, Note5, now 6pro..I...",Positive,NEGATIVE
278,Redmi,Mahesh,5.0 out of 5 stars,on 21 September 2018,Others,I love mi,NaN,Redmi I love mi,Positive,POSITIVE
